In [ ]:
2.2 Mini-BPE learner

In [1]:
from collections import Counter

# Training words
training_data = """
low low low low low lowest lowest
newer newer newer newer newer newer
wider wider wider new new
""".split()

# Store each word as characters with an end marker
frequency_table = Counter(
    tuple(item) + ("_",) for item in training_data
)

merge_history = []

# Initial set of symbols
symbol_vocabulary = {
    character
    for word in frequency_table
    for character in word
}


def get_pair_frequency(words):
    pair_frequency = Counter()

    for sequence, freq in words.items():
        for first, second in zip(sequence, sequence[1:]):
            pair_frequency[(first, second)] += freq

    return pair_frequency


def combine_pair(target_pair, words):
    updated_words = Counter()
    combined = "".join(target_pair)

    for sequence, freq in words.items():
        new_sequence = []
        position = 0

        while position < len(sequence):
            current_match = (
                position + 1 < len(sequence)
                and (sequence[position], sequence[position + 1]) == target_pair
            )

            if current_match:
                new_sequence.append(combined)
                position += 2
            else:
                new_sequence.append(sequence[position])
                position += 1

        updated_words[tuple(new_sequence)] += freq

    return updated_words


# Learn 10 BPE merge operations
for iteration in range(10):
    pair_frequency = get_pair_frequency(frequency_table)

    most_common_pair = max(
        pair_frequency,
        key=pair_frequency.get
    )

    frequency_table = combine_pair(
        most_common_pair,
        frequency_table
    )

    merge_history.append(most_common_pair)
    symbol_vocabulary.add("".join(most_common_pair))

    print(
        f"Step {iteration + 1}: {most_common_pair} "
        f"count={pair_frequency[most_common_pair]}, "
        f"vocab size={len(symbol_vocabulary)}"
    )


def bpe_tokenize(text):
    pieces = list(text) + ["_"]

    for learned_pair in merge_history:
        result = []
        index = 0

        while index < len(pieces):
            if (
                index + 1 < len(pieces)
                and (pieces[index], pieces[index + 1]) == learned_pair
            ):
                result.append("".join(learned_pair))
                index += 2
            else:
                result.append(pieces[index])
                index += 1

        pieces = result

    return pieces


# Test the learned BPE model
test_words = [
    "new",
    "newer",
    "lowest",
    "widest",
    "newestest"
]

for item in test_words:
    print(item, "->", bpe_tokenize(item))



Step 1: ('e', 'r') count=9, vocab size=12
Step 2: ('er', '_') count=9, vocab size=13
Step 3: ('n', 'e') count=8, vocab size=14
Step 4: ('ne', 'w') count=8, vocab size=15
Step 5: ('l', 'o') count=7, vocab size=16
Step 6: ('lo', 'w') count=7, vocab size=17
Step 7: ('new', 'er_') count=6, vocab size=18
Step 8: ('low', '_') count=5, vocab size=19
Step 9: ('w', 'i') count=3, vocab size=20
Step 10: ('wi', 'd') count=3, vocab size=21
new -> ['new', '_']
newer -> ['newer_']
lowest -> ['low', 'e', 's', 't', '_']
widest -> ['wid', 'e', 's', 't', '_']
newestest -> ['new', 'e', 's', 't', 'e', 's', 't', '_']


In [ ]:
2.3 BPE on english paragrph

In [2]:
from collections import Counter
import re

# A small training text for learning common character patterns.
text_data = """
Natural language processing helps computers understand human language. 
Subword tokenization breaks words into smaller pieces that a model can reuse. 
These pieces help the system handle unfamiliar words and different word forms. 
For example, researchers can study how computers process meaningful linguistic patterns.
A carefully trained tokenizer can represent rare vocabulary without treating every new word as unknown.
"""

# Extract simple English words and ignore capitalization.
training_words = re.findall(r"[a-z]+", text_data.lower())

# Represent every word as individual characters.
# "_" marks where the word ends.
word_table = Counter(
    tuple(term) + ("_",)
    for term in training_words
)

# Keep track of the symbols that existed before any BPE operations.
base_symbols = set()
for sequence in word_table:
    base_symbols.update(sequence)

# This list will contain the BPE rules learned from the corpus.
bpe_rules = []


def find_frequent_pairs(data):
    """Count how often each neighboring symbol pair occurs."""
    pair_frequency = Counter()

    for sequence, occurrences in data.items():
        adjacent_symbols = zip(sequence, sequence[1:])

        for symbol_pair in adjacent_symbols:
            pair_frequency[symbol_pair] += occurrences

    return pair_frequency


def apply_bpe_merge(data, selected_pair):
    """Replace every occurrence of the selected pair with one symbol."""
    updated_table = Counter()
    combined_symbol = "".join(selected_pair)

    for sequence, occurrences in data.items():
        rebuilt_sequence = []
        position = 0

        while position < len(sequence):

            # Check whether the current and next symbols form
            # the pair selected by the BPE algorithm.
            if (
                position + 1 < len(sequence)
                and (sequence[position], sequence[position + 1])
                == selected_pair
            ):
                rebuilt_sequence.append(combined_symbol)
                position += 2
            else:
                rebuilt_sequence.append(sequence[position])
                position += 1

        updated_table[tuple(rebuilt_sequence)] += occurrences

    return updated_table


# Repeatedly find the most common pair and merge it.
# The assignment requires at least 30 merge operations.
for merge_number in range(30):

    pair_counts = find_frequent_pairs(word_table)

    # Stop safely if there are no more adjacent pairs available.
    if len(pair_counts) == 0:
        break

    most_common = max(
        pair_counts,
        key=pair_counts.get
    )

    occurrence_count = pair_counts[most_common]
    merged_name = "".join(most_common)

    # Save the information so it can be used later for analysis.
    bpe_rules.append({
        "pair": most_common,
        "token": merged_name,
        "frequency": occurrence_count
    })

    word_table = apply_bpe_merge(
        word_table,
        most_common
    )

    print(
        f"Merge {merge_number + 1}: "
        f"{most_common[0]} + {most_common[1]} -> {merged_name} "
        f"(frequency = {occurrence_count})"
    )


# Find the five merge rules that occurred most frequently.
most_used_rules = sorted(
    bpe_rules,
    key=lambda rule: rule["frequency"],
    reverse=True
)[:5]

print("\nTop five most frequent merges:")
for rule in most_used_rules:
    first, second = rule["pair"]

    print(
        f"{first} + {second} -> {rule['token']} "
        f"(frequency = {rule['frequency']})"
    )


# Combine the original characters with the newly created BPE tokens.
known_tokens = base_symbols.union(
    rule["token"] for rule in bpe_rules
)

# Sort primarily by token length so that longer learned pieces appear first.
longest_pieces = sorted(
    known_tokens,
    key=lambda piece: (-len(piece), piece)
)[:5]

print("\nFive longest learned subword tokens:")
for piece in longest_pieces:
    print(piece)


def tokenize_with_bpe(word, rules):
    """Apply the learned merge rules to a new word."""
    current_pieces = list(word.lower()) + ["_"]

    for rule in rules:
        target_pair = rule["pair"]
        replacement = rule["token"]

        rebuilt = []
        position = 0

        while position < len(current_pieces):

            if (
                position + 1 < len(current_pieces)
                and (
                    current_pieces[position],
                    current_pieces[position + 1]
                ) == target_pair
            ):
                rebuilt.append(replacement)
                position += 2
            else:
                rebuilt.append(current_pieces[position])
                position += 1

        current_pieces = rebuilt

    return current_pieces


# Test both common words and words that may contain unfamiliar forms.
words_to_check = [
    "language",
    "tokenization",
    "uncommon",
    "inflected",
    "learners"
]

print("\nBPE word segmentation:")
for term in words_to_check:
    pieces = tokenize_with_bpe(term, bpe_rules)
    print(f"{term} -> {' | '.join(pieces)}")

Merge 1: s + _ -> s_ (frequency = 13)
Merge 2: a + n -> an (frequency = 10)
Merge 3: e + r -> er (frequency = 9)
Merge 4: e + _ -> e_ (frequency = 8)
Merge 5: o + r -> or (frequency = 7)
Merge 6: r + e -> re (frequency = 7)
Merge 7: i + n -> in (frequency = 6)
Merge 8: d + _ -> d_ (frequency = 6)
Merge 9: a + t -> at (frequency = 5)
Merge 10: w + or -> wor (frequency = 5)
Merge 11: c + e -> ce (frequency = 4)
Merge 12: in + g -> ing (frequency = 4)
Merge 13: s + t -> st (frequency = 4)
Merge 14: an + _ -> an_ (frequency = 4)
Merge 15: e + n -> en (frequency = 4)
Merge 16: t + h -> th (frequency = 4)
Merge 17: a + r -> ar (frequency = 4)
Merge 18: y + _ -> y_ (frequency = 4)
Merge 19: l + _ -> l_ (frequency = 3)
Merge 20: m + p -> mp (frequency = 3)
Merge 21: u + t -> ut (frequency = 3)
Merge 22: er + s_ -> ers_ (frequency = 3)
Merge 23: u + n -> un (frequency = 3)
Merge 24: wor + d_ -> word_ (frequency = 3)
Merge 25: t + o -> to (frequency = 3)
Merge 26: c + an_ -> can_ (frequency = 3)

In [ ]:
Q5 Naive tokenization code

In [3]:
paragraph = (
    "Ravi prathi roju udayam tvaraga lechi tana panulanu prarambhistadu."
    "Atanu munduga pustakalu chadivi, taruvatha college ki veltadu. "
    "Kotha vishayalanu nerchukovadam ante ataniki chala ishtam. "
   " Computer science lo programming mariyu technology gurinchi telusukovadaniki atanu ekkuva samayam ketaayistadu."
    "Snehitulato kalisi charchalu jaripi, kashtamaina samasyalaku parishkaralu kanugonadaniki prayatnistadu. "
   " Krushi mariyu pattudala unte ye lakshyanaina saadhinchavachani Ravi nammutadu."
)

naive_tokens = paragraph.split()

print("Naive space-based tokens:")
for token in naive_tokens:
    print(token)

Naive space-based tokens:
Ravi
prathi
roju
udayam
tvaraga
lechi
tana
panulanu
prarambhistadu.Atanu
munduga
pustakalu
chadivi,
taruvatha
college
ki
veltadu.
Kotha
vishayalanu
nerchukovadam
ante
ataniki
chala
ishtam.
Computer
science
lo
programming
mariyu
technology
gurinchi
telusukovadaniki
atanu
ekkuva
samayam
ketaayistadu.Snehitulato
kalisi
charchalu
jaripi,
kashtamaina
samasyalaku
parishkaralu
kanugonadaniki
prayatnistadu.
Krushi
mariyu
pattudala
unte
ye
lakshyanaina
saadhinchavachani
Ravi
nammutadu.
